# NeuroScan GPU Training — Kaggle
Trains both models and saves `.keras` checkpoints to `/kaggle/working/` for download.

**Datasets required (add via the right-side panel):**
1. `Brain Tumor MRI Dataset` — https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
2. `LGG Segmentation Dataset` — https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation

In [ ]:
import os, glob, cv2, numpy as np, tensorflow as tf
from sklearn.model_selection import train_test_split

# ── GPU check ─────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {gpus}')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'Training on GPU: {gpus[0]}')
else:
    print('WARNING: No GPU found — training on CPU')

# ── Paths ──────────────────────────────────────────────────────────────────────
CLS_TRAIN = '/kaggle/input/brain-tumor-mri-dataset/Training'
CLS_TEST  = '/kaggle/input/brain-tumor-mri-dataset/Testing'
SEG_DIR   = '/kaggle/input/lgg-mri-segmentation/kaggle_3m'
OUT_DIR   = '/kaggle/working'

# ── Config ─────────────────────────────────────────────────────────────────────
IMG_SIZE_SEG = (224, 224)
IMG_SIZE_CLS = (224, 224)
CLASSES      = ['glioma', 'meningioma', 'pituitary', 'notumor']
SEG_EPOCHS   = 50
CLS_EPOCHS   = 40
BATCH        = 16   # larger batch fits on T4 / P100
LR           = 5e-4

print('Setup complete.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SEGMENTATION DATA PIPELINE
# LGG dataset: each patient folder contains paired images + *_mask.tif files
# ═══════════════════════════════════════════════════════════════════════════════

def load_seg_pair(img_path, mask_path):
    # Image — grayscale, normalised [0,1]
    img  = tf.io.read_file(img_path)
    img  = tf.image.decode_png(img, channels=1)  # LGG TIF already converted by Kaggle
    img  = tf.image.convert_image_dtype(img, tf.float32)
    img  = tf.image.resize(img, IMG_SIZE_SEG)
    # Mask — binary
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.convert_image_dtype(mask, tf.float32)
    mask = tf.image.resize(mask, IMG_SIZE_SEG, method='nearest')
    mask = tf.cast(mask > 0.5, tf.float32)
    return img, mask

def augment_seg(img, mask):
    seed = tf.random.uniform(shape=[], minval=0, maxval=2**31-1, dtype=tf.int32)
    img  = tf.image.stateless_random_flip_left_right(img,  seed=[seed, 0])
    mask = tf.image.stateless_random_flip_left_right(mask, seed=[seed, 0])
    img  = tf.image.stateless_random_flip_up_down(img,  seed=[seed+1, 0])
    mask = tf.image.stateless_random_flip_up_down(mask, seed=[seed+1, 0])
    img  = tf.image.random_brightness(img, max_delta=0.1)
    img  = tf.clip_by_value(img, 0, 1)
    return img, mask

# Discover image/mask pairs
all_tifs = glob.glob(os.path.join(SEG_DIR, '**', '*.tif'), recursive=True)
img_paths, mask_paths = [], []
for p in all_tifs:
    if 'mask' not in p.lower():
        base, ext = os.path.splitext(p)
        msk = f'{base}_mask{ext}'
        if os.path.exists(msk):
            img_paths.append(p)
            mask_paths.append(msk)

print(f'Found {len(img_paths)} segmentation pairs')

# Convert TIF → PNG on-the-fly (Kaggle can read TIF directly via PIL)
# We'll use cv2 to pre-convert
png_imgs, png_masks = [], []
for ip, mp in zip(img_paths, mask_paths):
    # save as png to /kaggle/working/seg_data/
    os.makedirs('/kaggle/working/seg_data', exist_ok=True)
    fn = os.path.basename(ip).replace('.tif', '.png')
    fn_m = os.path.basename(mp).replace('.tif', '.png')
    png_p = f'/kaggle/working/seg_data/{fn}'
    png_m = f'/kaggle/working/seg_data/{fn_m}'
    if not os.path.exists(png_p):
        cv2.imwrite(png_p, cv2.imread(ip, cv2.IMREAD_UNCHANGED))
    if not os.path.exists(png_m):
        cv2.imwrite(png_m, cv2.imread(mp, cv2.IMREAD_UNCHANGED))
    png_imgs.append(png_p)
    png_masks.append(png_m)

print(f'Converted {len(png_imgs)} pairs to PNG')

# Train/val split
ti, vi, tm, vm = train_test_split(png_imgs, png_masks, test_size=0.15, random_state=42)
print(f'Train: {len(ti)}  Val: {len(vi)}')

def make_seg_ds(imgs, masks, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((imgs, masks))
    ds = ds.map(load_seg_pair, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_seg, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.shuffle(500)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

seg_train_ds = make_seg_ds(ti, tm, augment=True)
seg_val_ds   = make_seg_ds(vi, vm)
print('Seg datasets ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ATTENTION U-NET (SeparableConv2D)
# Matches the architecture in neuroscan/seg_model.py
# ═══════════════════════════════════════════════════════════════════════════════
from tensorflow.keras import layers, models

def conv_block(x, filters):
    x = layers.SeparableConv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.SeparableConv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x

def attention_gate(x, g, inter):
    tx = layers.Conv2D(inter, 1, strides=2, padding='same', use_bias=False)(x)
    pg = layers.Conv2D(inter, 1, padding='same', use_bias=False)(g)
    out = layers.Activation('relu')(layers.Add()([tx, pg]))
    psi = layers.Activation('sigmoid')(layers.Conv2D(1, 1, padding='same', use_bias=False)(out))
    psi = layers.UpSampling2D(size=2, interpolation='bilinear')(psi)
    return layers.Multiply()([x, psi])

def build_unet(input_shape=(224, 224, 1), filters=[16, 32, 64, 128, 256]):
    inputs = layers.Input(shape=input_shape)
    skips, x = [], inputs
    for f in filters[:-1]:
        x = conv_block(x, f)
        skips.append(x)
        x = layers.MaxPooling2D(2)(x)
    x = conv_block(x, filters[-1])
    for f, skip in zip(reversed(filters[:-1]), reversed(skips)):
        g = x
        x = layers.UpSampling2D(size=2, interpolation='bilinear')(x)
        x = layers.SeparableConv2D(f, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        skip = attention_gate(skip, g, f // 2)
        x = conv_block(layers.Concatenate()([x, skip]), f)
    outputs = layers.Conv2D(1, 1, padding='same', activation='sigmoid')(x)
    return models.Model(inputs, outputs, name='neuroscan_unet_sepconv')

seg_model = build_unet()
seg_model.summary(line_length=100)
print('U-Net built.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRAIN SEGMENTATION MODEL
# ═══════════════════════════════════════════════════════════════════════════════
import tensorflow.keras.backend as K

def dice_coef(y_true, y_pred, smooth=1.):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    return tf.keras.losses.binary_crossentropy(y_true, y_pred) + (1. - dice_coef(y_true, y_pred))

SEG_CKPT = os.path.join(OUT_DIR, 'neuroscan_seg.keras')

seg_model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss=bce_dice_loss,
    metrics=['accuracy', dice_coef, tf.keras.metrics.MeanIoU(num_classes=2)]
)

seg_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(SEG_CKPT, save_best_only=True, monitor='val_loss', verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
]

print('Starting Attention U-Net training...')
seg_history = seg_model.fit(
    seg_train_ds,
    validation_data=seg_val_ds,
    epochs=SEG_EPOCHS,
    callbacks=seg_callbacks,
)
print(f'Segmentation training complete. Best model saved to: {SEG_CKPT}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CLASSIFICATION DATA PIPELINE
# Brain Tumor MRI Dataset: Training/ and Testing/ folders, each with class subfolders
# ═══════════════════════════════════════════════════════════════════════════════

def load_cls_image(img_path, label):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE_CLS)
    img = tf.cast(img, tf.float32)  # [0, 255] — EfficientNetV2 handles its own normalisation
    return img, label

def augment_cls(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.clip_by_value(img, 0, 255)
    return img, label

def build_cls_ds(root, augment=False):
    paths, labels = [], []
    for idx, cls in enumerate(CLASSES):
        folder = os.path.join(root, cls)
        imgs = glob.glob(os.path.join(folder, '*.jpg')) + glob.glob(os.path.join(folder, '*.png'))
        paths.extend(imgs)
        labels.extend([idx] * len(imgs))
    print(f'  {root}: {len(paths)} images')
    cat_labels = tf.keras.utils.to_categorical(labels, num_classes=len(CLASSES))
    ds = tf.data.Dataset.from_tensor_slices((paths, cat_labels))
    ds = ds.map(load_cls_image, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_cls, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.shuffle(2000)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

cls_train_ds = build_cls_ds(CLS_TRAIN, augment=True)
cls_val_ds   = build_cls_ds(CLS_TEST)
print('Classification datasets ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EFFICIENTNETV2B0 CLASSIFIER
# Matches the architecture in neuroscan/cls_model.py
# ═══════════════════════════════════════════════════════════════════════════════
from tensorflow.keras.applications import EfficientNetV2B0

def build_classifier(input_shape=(224, 224, 3), num_classes=4):
    base = EfficientNetV2B0(include_top=False, weights='imagenet', input_shape=input_shape)
    base.trainable = False
    inputs  = layers.Input(shape=input_shape)
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs, name='neuroscan_efficientnet_cls')

cls_model = build_classifier()
cls_model.summary(line_length=100)
print('Classifier built.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 1: Train head only (base frozen)
# ═══════════════════════════════════════════════════════════════════════════════
CLS_CKPT = os.path.join(OUT_DIR, 'neuroscan_cls.keras')

cls_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(CLS_CKPT, save_best_only=True, monitor='val_loss', verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
]

cls_model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Phase 1: Training classification head (base frozen)...')
cls_model.fit(
    cls_train_ds,
    validation_data=cls_val_ds,
    epochs=CLS_EPOCHS,
    callbacks=cls_callbacks,
)
print('Phase 1 complete.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 2: Fine-tune top 20 layers of EfficientNetV2B0
# ═══════════════════════════════════════════════════════════════════════════════
base_model = cls_model.layers[1]  # EfficientNetV2B0
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

cls_model.compile(
    optimizer=tf.keras.optimizers.Adam(LR / 10),  # lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Phase 2: Fine-tuning top 20 layers...')
cls_model.fit(
    cls_train_ds,
    validation_data=cls_val_ds,
    epochs=CLS_EPOCHS // 2,
    callbacks=cls_callbacks,
)
print(f'Fine-tuning complete. Best model saved to: {CLS_CKPT}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SUMMARY — list outputs for download
# ═══════════════════════════════════════════════════════════════════════════════
print('\n=== Training Complete ===')
for f in [SEG_CKPT, CLS_CKPT]:
    size_mb = os.path.getsize(f) / 1e6 if os.path.exists(f) else 0
    print(f'  {os.path.basename(f):30s}  {size_mb:.1f} MB')

print('\nDownload both .keras files from the Output tab on the right.')
print('Place them in:  checkpoints/  inside your local project folder.')